위치인코딩

In [47]:
import math
import torch
from torch import nn

class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_len, dropout=0.1):
    super().__init__()
    self.dropout = nn.Dropout(p=dropout)
    position = torch.arange(max_len).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
    pe = torch.zeros(max_len, 1, d_model)
    pe[:, 0, 0::2] = torch.sin(position * div_term)
    pe[:, 0, 1::2] = torch.cos(position * div_term)
    self.register_buffer("pe", pe)

  def forward(self, x):
    return self.dropout(x + self.pe[: x.size(0)])

class TokenEmbedding(nn.Module):
  def __init__(self, vocab_size, emb_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, emb_size)
    self.emb_size = emb_size

  def forward(self, tokens):
    return self.embedding(tokens.long()) * math.sqrt(self.emb_size)

class Seq2SeqTransformer(nn.Module):
  def __init__(self, num_encoder_layers, num_decoder_layers, emb_size, max_len,
               nhead, src_vocab_size, tgt_vocab_size, dim_feedforward, dropout=0.1):
    super().__init__()
    self.src_tok_emb = TokenEmbedding(src_vocab_size, emb_size)
    self.tgt_tok_emb = TokenEmbedding(tgt_vocab_size, emb_size)
    self.positional_encoding = PositionalEncoding(emb_size, max_len, dropout)
    self.transformer = nn.Transformer(
        d_model=emb_size, nhead=nhead,
        num_encoder_layers=num_encoder_layers,
        num_decoder_layers=num_decoder_layers,
        dim_feedforward=dim_feedforward, dropout=dropout,
    )
    self.generator = nn.Linear(emb_size, tgt_vocab_size)

  def forward(self, src, tgt, src_mask, tgt_mask,
              src_padding_mask, tgt_padding_mask, memory_key_padding_mask):
    src_emb = self.positional_encoding(self.src_tok_emb(src))
    tgt_emb = self.positional_encoding(self.tgt_tok_emb(tgt))
    outs = self.transformer(
        src=src_emb, tgt=tgt_emb,
        src_mask=src_mask, tgt_mask=tgt_mask, memory_mask=None,
        src_key_padding_mask=src_padding_mask,
        tgt_key_padding_mask=tgt_padding_mask,
        memory_key_padding_mask=memory_key_padding_mask,
    )
    return self.generator(outs)

  def encode(self, src, src_mask):
    return self.transformer.encoder(self.positional_encoding(self.src_tok_emb(src)), src_mask)

  def decode(self, tgt, memory, tgt_mask):
    return self.transformer.decoder(self.positional_encoding(self.tgt_tok_emb(tgt)), memory, tgt_mask)

model = Seq2SeqTransformer(
    num_encoder_layers=3,
    num_decoder_layers=3,
    emb_size=512,
    max_len=512,
    nhead=8,
    src_vocab_size=len(vocab_transform[SRC_LANGUAGE]),
    tgt_vocab_size=len(vocab_transform[TGT_LANGUAGE]),
    dim_feedforward=512,
).to(DEVICE)

for p in model.parameters():
  if p.dim() > 1:
    nn.init.xavier_uniform_(p)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

print(sum(p.numel() for p in model.parameters()), "params")

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


33570389 params


In [48]:
!pip install datasets

In [49]:
!python -m spacy download de_core_news_sm
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 51.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 95.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [50]:
import spacy
from collections import Counter
from datasets import load_dataset

SRC_LANGUAGE, TGT_LANGUAGE = "de", "en"
UNK_IDX, PAD_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3
special_symbols = ["<unk>", "<pad>", "<bos>", "<eos>"]

nlp = {
    "de": spacy.load("de_core_news_sm"),
    "en": spacy.load("en_core_web_sm"),
}
token_transform = {
    lang: (lambda text, l=lang: [t.text for t in nlp[l].tokenizer(text)])
    for lang in nlp
}

class Vocab:
    def __init__(self, token_iter, min_freq=1):
        counter = Counter()
        for tokens in token_iter:
            counter.update(tokens)
        self.itos = special_symbols + [
            w for w, c in counter.most_common()
            if c >= min_freq and w not in special_symbols
        ]
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __call__(self, tokens):
        return [self.stoi.get(t, UNK_IDX) for t in tokens]

    def __len__(self):
        return len(self.itos)

    def get_itos(self):
        return self.itos

ds = load_dataset("bentrevett/multi30k")

vocab_transform = {
    lang: Vocab((token_transform[lang](x[lang]) for x in ds["train"]))
    for lang in [SRC_LANGUAGE, TGT_LANGUAGE]
}

print({k: len(v) for k, v in vocab_transform.items()})

{'de': 19214, 'en': 10837}


Transformer

In [51]:
import math
import torch
from torch import nn

class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_len, dropout=0.1):
    super().__init__()
    self.dropout = nn.Dropout(p=dropout)

    position = torch.arange(max_len).unsqueeze(1)
    div_term = torch.exp(
        torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
    )

    pe = torch.zeros(max_len, 1, d_model)
    pe[:, 0, 0::2] = torch.sin(position * div_term)
    pe[:, 0, 1::2] = torch.cos(position * div_term)
    self.register_buffer("pe", pe)

  def forward(self, x):
    x = x + self.pe[: x.size(0)]
    return self.dropout(x)

class TokenEmbedding(nn.Module):
  def __init__(self, vocab_size, emb_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, emb_size)
    self.emb_size = emb_size

  def forward(self, tokens):
    return self.embedding(tokens.long()) * math.sqrt(self.emb_size)

class Seq2SeqTransformer(nn.Module):
  def __init__(
      self,
      num_encoder_layers,
      num_decoder_layers,
      emb_size,
      max_len,
      nhead,
      src_vocab_size,
      tgt_vocab_size,
      dim_feedforward,
      dropout=0.1,
  ):
    super().__init__()
    self.src_tok_emb = TokenEmbedding(src_vocab_size, emb_size)
    self.tgt_tok_emb = TokenEmbedding(tgt_vocab_size, emb_size)
    self.positional_encoding = PositionalEncoding(
        d_model=emb_size, max_len=max_len, dropout=dropout
    )
    self.transformer = nn.Transformer(
        d_model=emb_size,
        nhead=nhead,
        num_encoder_layers=num_encoder_layers,
        num_decoder_layers=num_decoder_layers,
        dim_feedforward=dim_feedforward,
        dropout=dropout,
    )
    self.generator = nn.Linear(emb_size, tgt_vocab_size)

  def forward(self, src, tgt, src_mask, tgt_mask, src_padding_mask, tgt_padding_mask, memory_key_padding_mask):
    src_emb = self.positional_encoding(self.src_tok_emb(src))
    tgt_emb = self.positional_encoding(self.tgt_tok_emb(tgt))
    outs = self.transformer(
        src=src_emb,
        tgt=tgt_emb,
        src_mask=src_mask,
        tgt_mask=tgt_mask,
        memory_mask=None,
        src_key_padding_mask=src_padding_mask,
        tgt_key_padding_mask=tgt_padding_mask,
        memory_key_padding_mask=memory_key_padding_mask
    )
    return self.generator(outs)

  def encode(self, src, src_mask):
    return self.transformer.encoder(self.positional_encoding(self.src_tok_emb(src)), src_mask)

  def decode(self, tgt, memory, tgt_mask):
    return self.transformer.decoder(self.positional_encoding(self.tgt_tok_emb(tgt)), memory, tgt_mask)

In [52]:
import torch
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

BATCH_SIZE = 128

def sequential_transforms(*transforms):
  def func(txt_input):
    for transform in transforms:
      txt_input = transform(txt_input)
    return txt_input
  return func

def input_transform(token_ids):
  return torch.cat(
      (torch.tensor([BOS_IDX]), torch.tensor(token_ids), torch.tensor([EOS_IDX]))
  )

def collator(batch):
  src_batch, tgt_batch = [], []
  for src_sample, tgt_sample in batch:
    src_batch.append(text_transform[SRC_LANGUAGE](src_sample.rstrip("\n")))
    tgt_batch.append(text_transform[TGT_LANGUAGE](tgt_sample.rstrip("\n")))

  src_batch = pad_sequence(src_batch, padding_value=PAD_IDX)
  tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX)
  return src_batch, tgt_batch

text_transform = {}
for language in [SRC_LANGUAGE, TGT_LANGUAGE]:
  text_transform[language] = sequential_transforms(
      token_transform[language], vocab_transform[language], input_transform
  )

data_iter = [(x[SRC_LANGUAGE], x[TGT_LANGUAGE]) for x in ds["validation"]]
dataloader = DataLoader(data_iter, batch_size=BATCH_SIZE, collate_fn=collator)
source_tensor, target_tensor = next(iter(dataloader))

print("(source, target):")
print(next(iter(data_iter)))

print("source_batch:", source_tensor.shape)
print(source_tensor)

print("target_batch:", target_tensor.shape)
print(target_tensor)

(source, target):
('Eine Gruppe von Männern lädt Baumwolle auf einen Lastwagen', 'A group of men are loading cotton onto a truck')
source_batch: torch.Size([35, 128])
tensor([[   2,    2,    2,  ...,    2,    2,    2],
        [  14,    5,    5,  ...,    5,   21,    5],
        [  38,   12,   35,  ...,   12, 1775,   69],
        ...,
        [   1,    1,    1,  ...,    1,    1,    1],
        [   1,    1,    1,  ...,    1,    1,    1],
        [   1,    1,    1,  ...,    1,    1,    1]])
target_batch: torch.Size([30, 128])
tensor([[   2,    2,    2,  ...,    2,    2,    2],
        [   6,    6,    6,  ...,  253,   19,    6],
        [  39,   12,   35,  ...,   12, 3093,   61],
        ...,
        [   1,    1,    1,  ...,    1,    1,    1],
        [   1,    1,    1,  ...,    1,    1,    1],
        [   1,    1,    1,  ...,    1,    1,    1]])


In [53]:
import torch
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

BATCH_SIZE = 128

def sequential_transforms(*transforms):
  def func(txt_input):
    for transform in transforms:
      txt_input = transform(txt_input)
    return txt_input
  return func

def input_transform(token_ids):
  return torch.cat(
      (torch.tensor([BOS_IDX]), torch.tensor(token_ids), torch.tensor([EOS_IDX]))
  )

def collator(batch):
  src_batch, tgt_batch = [], []
  for src_sample, tgt_sample in batch:
    src_batch.append(text_transform[SRC_LANGUAGE](src_sample.rstrip("\n")))
    tgt_batch.append(text_transform[TGT_LANGUAGE](tgt_sample.rstrip("\n")))

  src_batch = pad_sequence(src_batch, padding_value=PAD_IDX)
  tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX)
  return src_batch, tgt_batch

text_transform = {}
for language in [SRC_LANGUAGE, TGT_LANGUAGE]:
  text_transform[language] = sequential_transforms(
      token_transform[language], vocab_transform[language], input_transform
  )

# Multi30k(split="valid") 대신 HF 데이터 사용
data_iter = [(x[SRC_LANGUAGE], x[TGT_LANGUAGE]) for x in ds["validation"]]
dataloader = DataLoader(data_iter, batch_size=BATCH_SIZE, collate_fn=collator)
source_tensor, target_tensor = next(iter(dataloader))

print("(source, target):")
print(next(iter(data_iter)))

print("source_batch:", source_tensor.shape)
print(source_tensor)

print("target_batch:", target_tensor.shape)
print(target_tensor)

(source, target):
('Eine Gruppe von Männern lädt Baumwolle auf einen Lastwagen', 'A group of men are loading cotton onto a truck')
source_batch: torch.Size([35, 128])
tensor([[   2,    2,    2,  ...,    2,    2,    2],
        [  14,    5,    5,  ...,    5,   21,    5],
        [  38,   12,   35,  ...,   12, 1775,   69],
        ...,
        [   1,    1,    1,  ...,    1,    1,    1],
        [   1,    1,    1,  ...,    1,    1,    1],
        [   1,    1,    1,  ...,    1,    1,    1]])
target_batch: torch.Size([30, 128])
tensor([[   2,    2,    2,  ...,    2,    2,    2],
        [   6,    6,    6,  ...,  253,   19,    6],
        [  39,   12,   35,  ...,   12, 3093,   61],
        ...,
        [   1,    1,    1,  ...,    1,    1,    1],
        [   1,    1,    1,  ...,    1,    1,    1],
        [   1,    1,    1,  ...,    1,    1,    1]])


In [54]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [55]:
def generate_square_subsequent_mask(s):
  mask = (torch.triu(torch.ones((s,s), device=DEVICE))==1).transpose(0,1)
  mask = (
      mask.float()
      .masked_fill(mask==0, float("-inf"))
      .masked_fill(mask==1, float(0.0))
  )
  return mask
  mask = (torch.triu(torch.ones((s,s), device=DEVICE))==1).transpose(0,1)
  mask = (
      mask.float()
      .masked_fill(mask==0, float("-inf"))
      .masked_fill(mask==1, float(0.0))
  )
  return mask

def create_mask(src, tgt):
  src_seq_len = src.shape[0]
  tgt_seq_len = tgt.shape[0]

  tgt_mask = generate_square_subsequent_mask(tgt_seq_len)
  src_mask = torch.zeros((src_seq_len, src_seq_len), device=DEVICE).type(torch.bool)

  src_padding_mask = (src == PAD_IDX).transpose(0, 1)
  tgt_padding_mask = (tgt == PAD_IDX).transpose(0, 1)
  return src_mask, tgt_mask, src_padding_mask, tgt_padding_mask

target_input = target_tensor[:-1, :]
target_out = target_tensor[1:, :]

source_mask, target_mask, source_padding_mask, target_padding_mask = create_mask(
    source_tensor, target_input
)

print("source_mask:",source_mask.shape)
print(source_mask)
print("target_mask:",target_mask.shape)
print(target_mask)
print("source_padding_mask:", source_padding_mask.shape)
print(source_padding_mask)
print("target_padding_mask:", target_padding_mask.shape)
print(target_padding_mask)

source_mask: torch.Size([35, 35])
tensor([[False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        ...,
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False]], device='cuda:0')
target_mask: torch.Size([29, 29])
tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf,
         -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf,
         -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf,
         -inf, -inf, -inf,

In [56]:
def run(model, optimizer, criterion, split):
  is_train = (split == "train")
  model.train() if is_train else model.eval()

  data_iter = [(x[SRC_LANGUAGE], x[TGT_LANGUAGE]) for x in ds[split]]
  dataloader = DataLoader(
      data_iter, batch_size=BATCH_SIZE, collate_fn=collator, shuffle=is_train
  )

  losses = 0
  with torch.set_grad_enabled(is_train):
    for source_batch, target_batch in dataloader:
      source_batch = source_batch.to(DEVICE)
      target_batch = target_batch.to(DEVICE)

      target_input = target_batch[:-1, :]
      target_out = target_batch[1:, :]

      src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(
          source_batch, target_input
      )

      logits = model(
          src=source_batch,
          tgt=target_input,
          src_mask=src_mask,
          tgt_mask=tgt_mask,
          src_padding_mask=src_padding_mask,
          tgt_padding_mask=tgt_padding_mask,
          memory_key_padding_mask=src_padding_mask,
      )

      loss = criterion(
          logits.reshape(-1, logits.shape[-1]), target_out.reshape(-1)
      )

      if is_train:
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
      losses += loss.item()

  return losses / len(dataloader)

for epoch in range(5):
  train_loss = run(model, optimizer, criterion, "train")
  val_loss = run(model, optimizer, criterion, "validation")
  print(f"Epoch: {epoch+1}, Train loss: {train_loss:.3f}, Val loss: {val_loss:.3f}")

Epoch: 1, Train loss: 5.336, Val loss: 4.060
Epoch: 2, Train loss: 3.723, Val loss: 3.258
Epoch: 3, Train loss: 3.116, Val loss: 2.837
Epoch: 4, Train loss: 2.729, Val loss: 2.577
Epoch: 5, Train loss: 2.443, Val loss: 2.403


Results

In [57]:
def encode(self, src, src_mask):
  return self.transformer.encoder(
      self.positional_encoding(self.src_tok_emb(src)), src_mask
  )

def decode(self, tgt, memory, tgt_mask):
  return self.transformer.decoder(
      self.positional_encoding(self.tgt_tok_emb(tgt)), memory, tgt_mask
  )

type(model).encode = encode
type(model).decode = decode

In [58]:
test_transform = text_transform

def greedy_decode(model, source_tensor, source_mask, max_len, start_symbol):
  source_tensor = source_tensor.to(DEVICE)
  source_mask = source_mask.to(DEVICE)

  memory = model.encode(source_tensor, source_mask)
  ys = torch.ones(1, 1).fill_(start_symbol).type(torch.long).to(DEVICE)
  for i in range(max_len - 1):
    target_mask = generate_square_subsequent_mask(ys.size(0))
    target_mask = target_mask.type(torch.bool).to(DEVICE)

    out = model.decode(ys, memory, target_mask)
    out = out.transpose(0, 1)
    prob = model.generator(out[:, -1])
    _, next_word = torch.max(prob, dim=1)
    next_word = next_word.item()

    ys = torch.cat(
        [ys, torch.ones(1, 1).type_as(source_tensor).fill_(next_word)], dim=0
    )
    if next_word == EOS_IDX:
      break
  return ys

@torch.no_grad()
def translate(model, source_sentence):
  model.eval()
  source_tensor = test_transform[SRC_LANGUAGE](source_sentence).view(-1, 1)
  num_tokens = source_tensor.shape[0]
  src_mask = (torch.zeros(num_tokens, num_tokens)).type(torch.bool)
  tgt_tokens = greedy_decode(
      model, source_tensor, src_mask, max_len=num_tokens + 5, start_symbol=BOS_IDX
  ).flatten().tolist()
  itos = vocab_transform[TGT_LANGUAGE].get_itos()
  words = [itos[t] for t in tgt_tokens if t not in (BOS_IDX, EOS_IDX)]
  return " ".join(words)

output_oov = translate(model, "Eine Gruppe von Menschen steht vor einem Iglu .")
output = translate(model, "Eine Gruppe von Menschen steht vor einem Gebäude .")
print(output_oov)
print(output)

A group of people standing in front of a store .
A group of people standing in front of a building .
